# Fine-tuning de DistilBERT para sentimiento en español (3 clases)

Este notebook muestra un flujo mínimo replicable para clasificar frases en español en tres clases: **NEGATIVO**, **NEUTRAL** y **POSITIVO**.

El flujo cubre:

1. Instalación y verificación de librerías.
2. Creación de un dataset pequeño de ejemplo.
3. Exploración básica del dataset.
4. Limpieza de texto.
5. Tokenización con DistilBERT multilingüe.
6. Fine-tuning con Hugging Face `Trainer`.
7. Evaluación e inferencia con frases nuevas.
8. Guardado del modelo.
9. Endpoint opcional con FastAPI en Colab.

> Nota: el dataset es intencionalmente pequeño para demostrar el flujo. Para un modelo útil, conviene recolectar muchas más frases por clase y usar `train`, `validation` y `test` separados.

## 1. Instalar dependencias

Ejecuta esta celda en Google Colab. Si Colab pide reiniciar el entorno después de instalar paquetes, reinicia y vuelve a ejecutar desde el inicio.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn fastapi uvicorn pyngrok

## 2. Verificar entorno y GPU

La GPU no es obligatoria para este ejemplo pequeño, pero acelera el entrenamiento.

In [ ]:
import torch
import transformers
import datasets

print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets: {datasets.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Crear dataset de frases en español

Usaremos tres etiquetas:

- `0`: NEGATIVO
- `1`: NEUTRAL
- `2`: POSITIVO

La columna `texto` contiene la frase y la columna `label` contiene la clase numérica.

In [ ]:
import pandas as pd
from datasets import Dataset

label_names = {0: "NEGATIVO", 1: "NEUTRAL", 2: "POSITIVO"}

frases = [
    # POSITIVO
    {"texto": "Este producto es excelente, superó mis expectativas.", "label": 2},
    {"texto": "La calidad es muy buena y llegó a tiempo.", "label": 2},
    {"texto": "Me encanta, lo recomiendo ampliamente.", "label": 2},
    {"texto": "El servicio fue rápido, claro y amable.", "label": 2},
    {"texto": "Funciona perfecto y volvería a comprarlo.", "label": 2},

    # NEUTRAL
    {"texto": "El producto llegó ayer por la tarde.", "label": 1},
    {"texto": "La caja incluye manual y cable USB.", "label": 1},
    {"texto": "El precio es similar al de otras tiendas.", "label": 1},
    {"texto": "El paquete contiene tres piezas.", "label": 1},
    {"texto": "Funciona bien, pero llegó tarde.", "label": 1},

    # NEGATIVO
    {"texto": "Horrible, no funciona como se describe.", "label": 0},
    {"texto": "La calidad es mala y se rompió rápido.", "label": 0},
    {"texto": "No lo recomiendo, fue una decepción.", "label": 0},
    {"texto": "El servicio fue lento y poco claro.", "label": 0},
    {"texto": "Totalmente inaceptable, no compren esto.", "label": 0},
]

df = pd.DataFrame(frases)
df["label_nombre"] = df["label"].map(label_names)

dataset = Dataset.from_pandas(df[["texto", "label"]])

print(f"Total de muestras: {len(df)}")
print("Balance por etiqueta:")
print(df["label_nombre"].value_counts())
df

## 4. Exploración básica del dataset

Antes de entrenar, revisa si las clases están balanceadas, si hay duplicados y si las frases tienen longitudes razonables.

In [ ]:
df["num_palabras"] = df["texto"].str.split().str.len()
df["num_caracteres"] = df["texto"].str.len()
duplicados = df.duplicated(subset=["texto"]).sum()

print("1) Balance por etiqueta:")
for label, total in df["label"].value_counts().sort_index().items():
    print(f"   - {label_names[label]}: {total} frases")

print("
2) Longitud de frases:")
print(df[["num_palabras", "num_caracteres"]].describe().round(1))

print(f"
3) Duplicados exactos: {duplicados}")

print("
4) Ejemplos por clase:")
for label in sorted(label_names):
    ejemplo = df[df["label"] == label].iloc[0]
    print(f"   - [{label_names[label]}] {ejemplo['texto']}")

## 5. Preprocesamiento de texto

En Transformers conviene evitar limpiezas agresivas al inicio. Aquí hacemos una limpieza mínima: normalizar espacios. También se muestra una versión alternativa que quita acentos y puntuación para comparar, pero el flujo principal conserva el texto con acentos.

In [ ]:
import re
import unicodedata

emoji_map = {
    "😍": " emoji_positivo ",
    "😡": " emoji_negativo ",
    "😐": " emoji_neutral ",
}

def reemplazar_emojis(texto):
    for emoji, token in emoji_map.items():
        texto = texto.replace(emoji, token)
    return texto

def limpieza_minima(texto):
    texto = reemplazar_emojis(texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def quitar_acentos_y_puntuacion(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(ch for ch in texto if unicodedata.category(ch) != "Mn")
    texto = re.sub(r"[^a-zA-Z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

texto_demo = "¡Me encantó el producto 😍, llegó rápido y funciona bien!"
print("Texto original:", texto_demo)
print("Limpieza mínima:", limpieza_minima(texto_demo))
print("Sin acentos ni puntuación:", quitar_acentos_y_puntuacion(limpieza_minima(texto_demo)))

df_preprocesado = df.copy()
df_preprocesado["texto_limpio"] = df_preprocesado["texto"].apply(limpieza_minima)

dataset_preprocesado = Dataset.from_dict({
    "texto": df_preprocesado["texto_limpio"].tolist(),
    "label": df_preprocesado["label"].tolist(),
})

dataset_split = dataset_preprocesado.train_test_split(test_size=0.33, seed=42)

print("
Dataset actualizado con textos limpios")
print(f"Muestras: {len(dataset_preprocesado)}")
print(f"Train: {len(dataset_split['train'])} | Test: {len(dataset_split['test'])}")

## 6. Cargar tokenizador y modelo DistilBERT multilingüe

`num_labels=3` configura la cabeza de clasificación para producir tres logits: uno para NEGATIVO, uno para NEUTRAL y uno para POSITIVO.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-multilingual-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=label_names,
    label2id={nombre: idx for idx, nombre in label_names.items()},
)

print(f"Modelo cargado: {MODEL_NAME}")
print(f"Configurado para: {model.config.num_labels} clases")
print(model.config.id2label)

## 7. Tokenizar el dataset

El tokenizador convierte texto en IDs numéricos. La `attention_mask` marca con `1` los tokens válidos y con `0` el padding.

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["texto"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset_split.map(tokenize_function, batched=True)

muestra = tokenized_dataset["train"][0]
print("Dataset tokenizado exitosamente")
print(f"Columnas train: {tokenized_dataset['train'].column_names}")
print("
Ejemplo de muestra 0:")
print(f"  Texto original: {muestra['texto']}")
print(f"  Input IDs (primeros 10): {muestra['input_ids'][:10]}")
print(f"  Attention mask (primeros 10): {muestra['attention_mask'][:10]}")
print(f"  Etiqueta: {muestra['label']} ({label_names[muestra['label']]})")

## 8. Fine-tuning con Trainer

Con este dataset pequeño, el resultado solo demuestra el mecanismo. En un proyecto real se recomienda usar muchas más muestras, validación separada, matriz de confusión y seguimiento de experimentos.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import Trainer, TrainingArguments, set_seed

set_seed(42)

training_args = TrainingArguments(
    output_dir="./fine_tuned_spanish_sentiment_3clases",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=5,
    evaluation_strategy="epoch",
    save_strategy="no",
    seed=42,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(
            labels,
            preds,
            labels=list(label_names.keys()),
            average="macro",
            zero_division=0,
        ),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

print("Iniciando fine-tuning multiclase en español...")
trainer.train()
metricas_test = trainer.evaluate()

predicciones = trainer.predict(tokenized_dataset["test"])
y_true = predicciones.label_ids
y_pred = predicciones.predictions.argmax(axis=-1)

print("
Fine-tuning completado")
print(f"Accuracy test: {metricas_test['eval_accuracy']:.2f}")
print(f"F1 macro test: {metricas_test['eval_f1_macro']:.2f}")
print("
Reporte por clase:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(label_names.keys()),
    target_names=list(label_names.values()),
    zero_division=0,
))

## 9. Inferencia con frases nuevas

La función recibe una frase, aplica el mismo tokenizador, obtiene logits, los convierte a probabilidades con `softmax` y devuelve la clase más probable.

In [ ]:
def predecir_sentimiento(texto):
    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.eval()

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    probs = torch.softmax(logits, dim=0).detach().cpu().numpy()
    pred = int(np.argmax(probs))

    return {
        "texto": texto,
        "label": label_names[pred],
        "confianza": float(probs[pred]),
        "probabilidades": {label_names[i]: float(probs[i]) for i in range(len(probs))},
    }

frases_prueba = [
    "El producto es excelente y llegó antes de lo esperado.",
    "El paquete incluye manual y cable USB.",
    "Totalmente inaceptable, no compren esto.",
    "Funciona bien pero llegó tarde.",
]

for i, frase in enumerate(frases_prueba, 1):
    resultado = predecir_sentimiento(frase)
    print(f"
Ejemplo {i}:")
    print(f"  Texto: {resultado['texto']}")
    print(f"  Predicción: {resultado['label']}")
    print(f"  Confianza: {resultado['confianza']:.1%}")
    for clase, prob in resultado["probabilidades"].items():
        print(f"  Prob. {clase}: {prob:.1%}")

## 10. Guardar modelo y tokenizador

`save_pretrained` guarda los pesos, configuración y tokenizador en un formato que puede cargarse después desde otra sesión o desde una API.

In [ ]:
RUTA_MODELO = "./modelo_sentimiento_es_3clases"

model.save_pretrained(RUTA_MODELO)
tokenizer.save_pretrained(RUTA_MODELO)

print(f"Modelo guardado en: {RUTA_MODELO}")
print("Archivos esperados: config.json, model.safetensors/pytorch_model.bin, tokenizer.json, vocab.txt")

## 11. Endpoint FastAPI opcional en Colab

Colab puede ejecutar FastAPI localmente, pero para acceder desde fuera del notebook necesita un túnel como `ngrok`.

Para usar `ngrok`, puede ser necesario configurar un authtoken desde https://ngrok.com/. Si no se configura, esta parte puede fallar dependiendo de la cuenta y límites de ngrok.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

app = FastAPI(title="API de sentimiento en español")

tokenizer_api = AutoTokenizer.from_pretrained("./modelo_sentimiento_es_3clases")
model_api = AutoModelForSequenceClassification.from_pretrained("./modelo_sentimiento_es_3clases")
model_api.eval()

id2label_api = {0: "NEGATIVO", 1: "NEUTRAL", 2: "POSITIVO"}

class EntradaTexto(BaseModel):
    texto: str

@app.post("/predict")
def predict(entrada: EntradaTexto):
    inputs = tokenizer_api(
        entrada.texto,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    with torch.no_grad():
        logits = model_api(**inputs).logits[0]

    probs = torch.softmax(logits, dim=0)
    pred = torch.argmax(probs).item()

    return {
        "label": id2label_api[pred],
        "confianza": float(probs[pred]),
        "probabilidades": {
            "negativo": float(probs[0]),
            "neutral": float(probs[1]),
            "positivo": float(probs[2]),
        },
    }

print("API definida. Ejecuta la siguiente celda para levantar el servidor en Colab.")

## 12. Levantar FastAPI con ngrok

Ejecuta esta celda solo si quieres probar el endpoint desde una URL pública temporal.

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

# Si tienes token de ngrok, descomenta y reemplaza:
# ngrok.set_auth_token("TU_NGROK_AUTHTOKEN")

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

public_url = ngrok.connect(8000)
print("URL pública temporal:", public_url)
print("Endpoint:", f"{public_url}/predict")

## 13. Probar el endpoint desde Python

Reemplaza `URL_PUBLICA` por la URL generada por ngrok si quieres probar la API desde el mismo notebook.

In [ ]:
import requests

# Ejemplo:
# URL_PUBLICA = "https://xxxxx.ngrok-free.app"
# response = requests.post(
#     f"{URL_PUBLICA}/predict",
#     json={"texto": "El producto llegó tarde, pero funciona bien."},
# )
# print(response.json())